<a href="https://colab.research.google.com/github/iishikawalia/Machine-Learning/blob/main/basic_MNIST.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn

## Dataset

In [6]:
import torchvision
from torchvision import datasets
from torchvision.transforms import ToTensor

In [8]:
print(f"PyTorch version: {torch.__version__}\ntorchvision version: {torchvision.__version__}")


PyTorch version: 2.11.0+cpu
torchvision version: 0.26.0+cpu


In [9]:
# training data
train_data = datasets.MNIST(root="data",train=True,download=True,transform=ToTensor(),target_transform=None)

# testing data
test_data = datasets.FashionMNIST(root="data",train=False,download=True,transform=ToTensor())

100%|██████████| 9.91M/9.91M [00:00<00:00, 40.1MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 1.18MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 10.7MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 2.07MB/s]
100%|██████████| 26.4M/26.4M [00:01<00:00, 16.4MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 278kB/s]
100%|██████████| 4.42M/4.42M [00:00<00:00, 5.15MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 9.31MB/s]


In [11]:
print(train_data[0])

(tensor([[[0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
          0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
          0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
          0.0000, 0.0000, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
          0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
          0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
          0.0000, 0.0000, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
          0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
          0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
          0.0000, 0.0000, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
          0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
          0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000

In [16]:
image, label = train_data[0]
image.shape

torch.Size([1, 28, 28])

In [13]:
len(train_data.data), len(train_data.targets), len(test_data.data), len(test_data.targets)

(60000, 60000, 10000, 10000)

In [19]:
class_names = train_data.classes
class_names

['0 - zero',
 '1 - one',
 '2 - two',
 '3 - three',
 '4 - four',
 '5 - five',
 '6 - six',
 '7 - seven',
 '8 - eight',
 '9 - nine']

## Dataloader


In [20]:
from torch.utils.data import DataLoader

In [22]:
BATCH_SIZE=64
train_dataloader=DataLoader(train_data,batch_size=BATCH_SIZE,shuffle=True)
test_dataloader=DataLoader(test_data,batch_size=BATCH_SIZE,shuffle=False)

In [26]:
for images, labels in train_dataloader:
    print(images.shape)
    print(labels.shape)
    break

torch.Size([64, 1, 28, 28])
torch.Size([64])


In [28]:
for images, labels in train_dataloader:
    print(images[0].shape)
    print(labels[0])
    break

torch.Size([1, 28, 28])
tensor(6)


## Model

In [32]:
model=nn.Sequential(
    nn.Flatten(),
    nn.Linear(28*28,128),
    nn.ReLU(),
    nn.Linear(128,10)
)

In [34]:
for images, labels in train_dataloader:
    outputs = model(images)
    print(outputs.shape)
    break

torch.Size([64, 10])


In [35]:
for images, labels in train_dataloader:
    outputs = model(images[0])
    print(outputs.shape)
    break

torch.Size([1, 10])


## Loss

In [37]:
loss_fn = nn.CrossEntropyLoss() #standard loss for multi-class classification

## Optimizer

In [39]:
optimizer = torch.optim.SGD(params=model.parameters(), lr=0.1)

## Training

In [41]:
from timeit import default_timer as timer
def print_train_time(start: float, end: float, device: torch.device = None):
    total_time = end - start
    print(f"Train time on {device}: {total_time:.3f} seconds")
    return total_time

In [46]:
def accuracy_fn(y_true, y_pred):
    correct = torch.eq(y_true, y_pred).sum().item()
    acc = (correct/len(y_pred)) * 100
    return acc

In [67]:
for epoch in range(5):
    epoch_loss = 0
    correct = 0
    total = 0
    for images, labels in train_dataloader:
        outputs = model(images) # outputs.shape = torch.Size([64, 10])
        predictions = outputs.argmax(dim=1) # predictions.shape = torch.Size([64])
        correct += (predictions == labels).sum().item()
        total += labels.shape[0]
        loss = loss_fn(outputs, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    print(f"Epoch: {epoch} | Loss: {epoch_loss/len(train_dataloader)}")
    print(f"Last epoch loss: {loss.item()}")
    accuracy = correct/total
    print(f"Accuracy:{accuracy}")

Epoch: 0 | Loss: 0.003444148321167009
Last epoch loss: 0.0008761072531342506
Accuracy:0.9999833333333333
Epoch: 1 | Loss: 0.003350734280992988
Last epoch loss: 0.0023743100464344025
Accuracy:0.9999666666666667
Epoch: 2 | Loss: 0.003284308823634253
Last epoch loss: 0.0027128441724926233
Accuracy:0.99995
Epoch: 3 | Loss: 0.003164390793578366
Last epoch loss: 0.0006470330408774316
Accuracy:1.0
Epoch: 4 | Loss: 0.003062269073535702
Last epoch loss: 0.0021228170953691006
Accuracy:0.9999833333333333


In [66]:
for images, labels in train_dataloader:
    outputs = model(images)
    print(outputs.shape)
    predictions = outputs.argmax(dim=1)
    print(predictions.shape)
    print(f"Predicted: {predictions[:10]}")
    print(f"True Label: {labels[:10]}")
    break

torch.Size([64, 10])
torch.Size([64])
Predicted: tensor([8, 3, 3, 1, 9, 8, 1, 3, 4, 8])
True Label: tensor([8, 3, 3, 1, 9, 8, 1, 3, 4, 8])
